In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.4/233.4 kB 20.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 23.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 60.2 MB/s e

In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
from torch.optim.lr_scheduler import CosineAnnealingLR
# --- THE MAIN SUBJECT LOOP ---

import torch_geometric.loader as geom_loader # Special loader for graphs
import copy
import scipy.signal as signal
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
import numpy as np
import os

In [3]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'], 
    num_worker=0,
    offline_transform=transforms.Compose([
        transforms.To2d(), 
        transforms.Lambda(lambda x: torch.tensor(x).float())]),
    label_transform=transforms.Select('emotion'),
)

[2026-03-06 14:04:42] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-03-06 14:04:42] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 1it [00:00,  3.61it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 28it [00:00, 93.77it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 58it [00:00, 160.19it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 86it [00:00, 197.72it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 114it [00:00, 221.99it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_201511

In [5]:
def map_emotions(y):
    if y == 1 or y == 2:  # Sad or Fear
        return 0 
    if y == 3:  # Happy
        return 1  
    return -1 # Neutral

In [7]:
# This DataFrame contains columns like: subject_id, trial_id, emotion, etc.
meta_info = dataset.info 
all_subjects = sorted(meta_info['subject_id'].unique())
print(f"Subjects Found: {all_subjects}")

Subjects Found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    """Basic Graph Convolution Layer."""
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        # x: (B, N, Fin), adj: (B, N, N)
        support = torch.matmul(x, self.weight)
        output = torch.matmul(adj, support)
        return F.relu(output)

class MultiGraphicLayerConstruction(nn.Module):
    """Transformer-based adaptive adjacency matrix construction."""
    def __init__(self, feature_dim, num_heads=2, k=5):
        super(MultiGraphicLayerConstruction, self).__init__()
        self.k = k
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(feature_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, batch_first=True)
        
        self.norm2 = nn.LayerNorm(feature_dim)
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 2),
            nn.ReLU(),
            nn.Linear(feature_dim * 2, feature_dim)
        )

       # داخل كلاس MultiGraphicLayerConstruction
    def forward(self, x):
        B, N, D = x.shape
        x_norm = self.norm1(x)
        
        # 1. Scaled Dot-Product
        scaling = D ** 0.5
        _, attn_weights = self.multihead_attn(x_norm, x_norm, x_norm)
        
        attn_out, _ = self.multihead_attn(x_norm, x_norm, x_norm)
        x2 = self.norm2(x + attn_out)
        g_feature = self.mlp(x2) + x2
        
        # 2. Scaling الـ Matrix multiplication قبل الـ Softmax
        s_matrix = torch.matmul(g_feature, g_feature.transpose(-1, -2)) / scaling
        
        # 3. Adjacency Matrix
        adj = F.softmax(attn_weights + s_matrix, dim=-1)
        
        # 4. Top-K Sparsity (مهم جداً للـ Stability)
        # تأكد إن الـ K مش كبيرة أوي (10 مناسبة جداً لـ 62 قناة)
        topk_val, _ = torch.topk(adj, self.k, dim=-1)
        mask = (adj >= topk_val[:, :, -1].unsqueeze(-1)).float()
        adj = adj * mask
        
        return adj, g_feature

class BFENet_SingleBand(nn.Module):
    """Implementation of BFE-Net for one frequency band."""
    def __init__(self, num_channels=62):
        super(BFENet_SingleBand, self).__init__()
        
        # CNN Layer: Processes single-channel features independently
        # Input (B, 1, 1) -> Paper implies processing the DE feature dimension
        # Since input is (B, 62, 5), for a single band it's (B, 62, 1)
        self.cnn1 = nn.Conv1d(1, 64, kernel_size=1) 
        self.cnn2 = nn.Conv1d(64, 128, kernel_size=1)
        self.cnn3 = nn.Conv1d(128, 256, kernel_size=1)
        
        self.graph_const = MultiGraphicLayerConstruction(feature_dim=256)
        self.gcn = GCNLayer(256, 128)
        self.dropout = nn.Dropout(0)

    def forward(self, x):
        # x: (B, 62, 1) -> treat 62 as nodes, 1 as feature
        B, N, _ = x.shape
        x = x.transpose(1, 2) # (B, 1, 62)
        
        # CNN Feature Extraction
        x = F.relu(self.cnn1(x))
        x = F.relu(self.cnn2(x))
        x = F.relu(self.cnn3(x)) # (B, 256, 62)
        
        x = x.transpose(1, 2) # (B, 62, 256) -> (B, Nodes, Features)
        
        # Adaptive Graph Construction
        adj, g_features = self.graph_const(x)
        
        # Graph Convolution
        x_gcn = self.gcn(g_features, adj) # (B, 62, 128)
        
        # Flatten for fusion
        x_flat = x_gcn.reshape(B, -1) 
        return x_flat

class BFENet_Full(nn.Module):
    """The full model that fuses 5 frequency bands."""
    def __init__(self, num_classes=3): # 3 for SEED, 4 for SEED-IV
        super(BFENet_Full, self).__init__()
        self.bands = nn.ModuleList([BFENet_SingleBand() for _ in range(5)])
        
        # Final classification layer
        # Input: 5 bands * (62 nodes * 128 gcn features)
        self.classifier = nn.Sequential(
            nn.Linear(5 * 62 * 128, 256),
            nn.ReLU(),
            nn.Dropout(0),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, 62, 5)
        band_features = []
        for i in range(5):
            band_data = x[:, :, i].unsqueeze(-1) # (B, 62, 1)
            band_out = self.bands[i](band_data)
            band_features.append(band_out)
        
        # Fusion
        merged = torch.cat(band_features, dim=1)
        logits = self.classifier(merged)
        return logits



In [34]:
# Parameters
EPOCHS = 200
BATCH_SIZE = 64
LR = 1e-4
PATIENCE_LR = 3
REDUCE_FACTOR = 0.5
PATIENCE_ES = 20
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.1

In [27]:
# Subject-Dependent Loop in 3 class
from sklearn.model_selection import train_test_split

# Create a directory to save the best models
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')

subject_results = {}

subject_class_results = {} # To store per-class accuracy for each subject


print("Start Subject-Dependent Evaluation...")

for subject_id in all_subjects:
    print(f"\n{'='*30} Processing Subject {subject_id} {'='*30}")
    
    # Create a unique ID for each recording (session + trial)
    # Since SEED IV repeats trial IDs (1-24) across sessions, 
    # merging them ensures we recognize all 72 trials per subject
    # remove neutral
    subject_df = meta_info[meta_info['subject_id'] == subject_id].copy()
    subject_df['emotion_mapped'] = subject_df['emotion'].apply(map_emotions)
    subject_df = subject_df.reset_index(drop=True)
    subject_df['unique_run_id'] = subject_df['session_id'].astype(str) + "_" + subject_df['trial_id'].astype(str)
    

    # Extract labels per unique run to ensure balanced split
    # We get a single label for each unique run (session_trial)
    # remove neutral
    run_info = subject_df[['unique_run_id', 'emotion_mapped']].drop_duplicates()
    all_runs = run_info['unique_run_id'].values
    all_labels = run_info['emotion_mapped'].values

    
    # Stratified Split at the RUN level 
    # This keeps 80/20 ratio and ensures each class is represented in the test set
    train_runs, test_runs = train_test_split(
        all_runs, 
        test_size=0.20, 
        random_state=42, 
        stratify=all_labels
    )
    
    
    print(f"Total Unique Videos (Across 3 Sessions): {len(all_runs)}")
    print(f"Training on: {len(train_runs)} | Testing on: {len(test_runs)}")
    
    # Extract indices (Zero Leakage Guaranteed)
    train_indices = subject_df[subject_df['unique_run_id'].isin(train_runs)].index.tolist()
    test_indices = subject_df[subject_df['unique_run_id'].isin(test_runs)].index.tolist()


    # Dataset Subsets and Loaders
    train_set = Subset(dataset, train_indices)
    test_set = Subset(dataset, test_indices)
    
    # Weighted Sampler for further balancing during training 
    # remove neutral 
    y_train_mapped = subject_df.loc[train_indices, 'emotion_mapped'].values
    class_counts = np.bincount(y_train_mapped)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in y_train_mapped]


    
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, sampler=sampler,num_workers=0,pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,num_workers=0,pin_memory=True)


    # --- 3. Initialize Model ---
    model = BFENet_Full(num_classes=3).to(device)
    
    criterion = nn.CrossEntropyLoss( label_smoothing = LABEL_SMOOTHING)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
   
    # Training Loop 
    best_val_acc = 0.0
    counter = 0 # For early stopping
    
    for epoch in range(EPOCHS):
        model.train()
        correct = 0
        total = 0
        train_loss = 0
        
        for X, y in train_loader:
            X = X.to(device)
            
            y_bin = torch.where((y == 1) | (y == 2), 0, 
                    torch.where(y == 3, 1, 2)).to(device).long()
            
            
            if len(X.shape) == 4: X = X.squeeze(1) 

            mean = X.mean(dim=(1, 2), keepdim=True)
            std = X.std(dim=(1, 2), keepdim=True) + 1e-6
            X = (X - mean) / std
          
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y_bin)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += y_bin.size(0)
            correct += (predicted == y_bin).sum().item()
            
       
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * correct / total
            
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

      
        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device)
               
                y_bin = torch.where((y == 1) | (y == 2), 0, 
                    torch.where(y == 3, 1, 2)).to(device).long()
                
                if len(X.shape) == 4: X = X.squeeze(1)

                mean = X.mean(dim=(1, 2), keepdim=True)
                std = X.std(dim=(1, 2), keepdim=True) + 1e-6
                X = (X - mean) / std
                
                outputs = model(X)
                val_loss += criterion(outputs, y_bin).item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += y_bin.size(0)
                val_correct += (predicted == y_bin).sum().item()

                
        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(test_loader)
        scheduler.step()

        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")
        
        # Early Stopping Check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0

            save_path = f"saved_models/subject_{subject_id}_best.pth"
            torch.save(model.state_dict(), save_path)
            
        else:
            counter += 1
            if counter >= PATIENCE_ES:
                print(f"Early stopping at epoch {epoch}")
                break
                
    print(f"Subject {subject_id} Best Acc: {best_val_acc:.2f}%")
    subject_results[subject_id] = best_val_acc
   

# ================= FINAL REPORT =================
print("\n" + "="*50)
print("FINAL RESULTS REPORT ")
print("="*50)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
  
    print(f"Subject {sub_id:02d}: {subject_results[sub_id]:.2f}%")
    
# print("-" * 50)
print(f"Overall Average Accuracy: {np.mean(all_accuracies):.2f}%")

if len(subject_results) > 0:
    best_sub_id = max(subject_results, key=subject_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({subject_results[best_sub_id]:.2f}%)")
print("="*50)

Start Subject-Dependent Evaluation...

============================== Processing Subject 1 ==============================
Total Unique Videos (Across 3 Sessions): 72
Training on: 57 | Testing on: 15
Epoch 01 | Train Loss: 1.0667 Acc: 46.73% | Val Loss: 1.0708 Acc: 43.07%
Epoch 02 | Train Loss: 1.0601 Acc: 46.32% | Val Loss: 1.0675 Acc: 43.07%
Epoch 03 | Train Loss: 1.0512 Acc: 46.98% | Val Loss: 1.1066 Acc: 33.90%
Epoch 04 | Train Loss: 1.0341 Acc: 49.42% | Val Loss: 1.0731 Acc: 30.15%
Epoch 05 | Train Loss: 1.0327 Acc: 49.32% | Val Loss: 1.0764 Acc: 30.15%
Epoch 06 | Train Loss: 1.0492 Acc: 46.02% | Val Loss: 1.0829 Acc: 30.15%
Epoch 07 | Train Loss: 1.0200 Acc: 50.94% | Val Loss: 1.0871 Acc: 30.15%
Epoch 08 | Train Loss: 1.0377 Acc: 49.21% | Val Loss: 1.0765 Acc: 30.15%
Epoch 09 | Train Loss: 1.0107 Acc: 52.05% | Val Loss: 1.0861 Acc: 30.15%
Epoch 10 | Train Loss: 0.9966 Acc: 54.19% | Val Loss: 1.0040 Acc: 54.49%
Epoch 11 | Train Loss: 0.9530 Acc: 59.61% | Val Loss: 0.9535 Acc: 54.49

In [24]:
# ================= FINAL REPORT =================
print("\n" + "="*50)
print("FINAL RESULTS REPORT ")
print("="*50)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
  
    print(f"Subject {sub_id:02d}: {subject_results[sub_id]:.2f}%")
    
# print("-" * 50)
print(f"Overall Average Accuracy: {np.mean(all_accuracies):.2f}%")

if len(subject_results) > 0:
    best_sub_id = max(subject_results, key=subject_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({subject_results[best_sub_id]:.2f}%)")
print("="*50)


FINAL RESULTS REPORT 
Subject 01: 92.13%
Subject 02: 91.20%
Subject 03: 94.19%
Subject 04: 92.88%
Subject 05: 93.07%
Subject 06: 100.00%
Subject 07: 92.51%
Subject 08: 89.89%
Subject 09: 94.94%
Subject 10: 92.51%
Subject 11: 95.13%
Subject 12: 90.26%
Subject 13: 91.76%
Subject 14: 100.00%
Subject 15: 95.13%
Overall Average Accuracy: 93.71%
HIGHEST ACCURACY: Subject 6 (100.00%)
